# Imports and Installations

In [ ]:
#Downloading for audio
pip install -U yt-dlp
#For webscraping
pip install selenium
pip install webdriver_manager
pip install webdriver_manager

In [ ]:
#Imports
import subprocess
import os
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from webdriver_manager.chrome import ChromeDriverManager
import re
import csv
from tqdm import tqdm

In [ ]:
#Connecting to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Getting the urls using Selinium

In [ ]:
#Setting up Selenium with ChromeDriver for an endless scrolling page
options = Options()
options.add_argument('--headless')  #Running without opening a browser
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')

#Creating the Selenium webdriver instance
driver = webdriver.Chrome(options=options)

#URL of the page that I want to scrape - Emergency Meeting Playlist
#This page has infinite scrolling
url = 'https://rumble.com/playlists/f-Ue_cRBhOs'

#Telling the driver to navigate to the specific URL
driver.get(url)

#List to save video links
video_links = set()

#The time to let the new page load
scroll_pause_time = 3

#Recording the scrolling height
last_height = driver.execute_script('return document.body.scrollHeight')

while True:
    #Scrolling down to the bottom
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')

    #Pausing for the new content to load
    time.sleep(scroll_pause_time)

    #Extracting the video links
    video_elements = driver.find_elements(By.CSS_SELECTOR, 'a.title__link.link') #Finding links using CSS
    for video in video_elements:
        href = video.get_attribute('href') #Grabbing the url
        video_links.add(href) #Adding link to list

    #Checking if the page height has changed
    new_height = driver.execute_script('return document.body.scrollHeight')
    if new_height == last_height:
        break  # Stop - no new content is loading
    last_height = new_height

#Ending the scraper
driver.quit()

# Creating a CSV file of Links and Labels

In [ ]:
#Cleaning the links
cleaned_links = []

for link in video_links:
  cleaned_links.append(re.sub(r"\?.*", "", link))

#Sorting the links
cleaned_links.sort()
cleaned_links

In [ ]:
#Grabbing the video labels

#Regex pattern to match the label
pattern = r"(?<=emergency-meeting-)(.*?)(?=\.html)"

#Extracting the labels
labels = [re.search(pattern, url).group(0) for url in cleaned_links if re.search(pattern, url)]


In [ ]:
# Combining the lists
links_names = zip(cleaned_links, labels)

#Creating a DF
df = pd.DataFrame(links_names, columns=['Links', 'Labels'])

#Defining the file path to save the CSV
save_path ='/content/drive/MyDrive/Emergency_Meeting_Deepgram/'
csv_path = save_path + 'links_and_labels.csv'

# Saving the DF to a CSV file
df.to_csv(csv_path, index=False)


# Importing CSV file

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Emergency_Meeting_Deepgram/links_and_labels.csv')

# Creating the file paths for organization

In [ ]:
for label in df['Labels']:
  #Creating seperate files for each video - to save audio and text files
  save_path = f"/content/drive/MyDrive/Emergency_Meeting_Deepgram/{label}"

  #Checking if the directory exists
  os.makedirs(save_path, exist_ok=True)


# Downloading the mp3 audio

In [ ]:
#function to download audio using the links
def download_audio(url, output_path):
    try:
        command = ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', output_path, url] #Convert the audio to MP3
        subprocess.run(command, check=True)
        print(f"Downloaded audio to {output_path}")
    except subprocess.CalledProcessError as e: #If there is an error
        print('Error downloading audio:', e)

#Function to check in the downloaded audio file exsits and is valid
def check_audio_file(file_path):
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        print(f"File downloaded: {file_path}")
    else:
        print('Download failed')


In [ ]:
#Defining the Rumble video URL and output file
for i in range(86,102): #Because it was slow I ran the range for about 15 videos at a time
  rumble_url = df['Links'][i] #Grabbing the URL from the df
  audio_path = f"/content/drive/MyDrive/Emergency_Meeting_Deepgram/{df['Labels'][i]}/{df['Labels'][i]}.mp3"  #Audio file path

#Running the functions
  download_audio(rumble_url, audio_path)
  check_audio_file(audio_path)

# Deepgram Transcriptions


## Deepgram tokens needed to utalize Deepgram API

In [ ]:
#deepgram_key = "" #Removed the actual key

## Getting Transcriptions with Confidence Score

In [ ]:
#Creating a function that takes the audio and transcribes it to text
#Returning a json file with confidence scores and text and a plain text file
def transcribe_audio(file_path):
    url = 'https://api.deepgram.com/v1/listen?punctuate=true' #having deepgram put in punctuation
    headers = {'Authorization': f"Token {deepgram_key}"}

    with open(file_path, 'rb') as audio_file:
        response = requests.post(url, headers=headers, files={'file': audio_file})

        if response.status_code == 200:
          transcript_data = response.json()  #Getting the transcription data

          #Gettin the transcript
          full_transcript = transcript_data.get('results', {}).get('channels', [{}])[0].get('alternatives', [{}])[0].get('transcript', '')

          #Gettin the confidence score
          total_confidence = transcript_data.get('results', {}).get('channels', [{}])[0].get('alternatives', [{}])[0].get('confidence', '')

          #Saving as a JSON file
          transcript_output = {
              'transcript': full_transcript,
              'total_confidence': total_confidence
          }
          return transcript_output, full_transcript
        else:
          print('error:', response.text)


In [ ]:
#Checking with one file
index = 0

#Audio file path
audio_path = f"/content/drive/MyDrive/Emergency_Meeting_Deepgram/{df['Labels'][index]}/{df['Labels'][index]}.mp3"

transcript_json, transcript_txt = transcribe_audio(audio_path)

## Saving TEXT and JSON to file path


In [ ]:
#Chunked the index range because doing the full 102 was too much for my computer
for index in range(50,102):
  audio_path = f"/content/drive/MyDrive/Emergency_Meeting_Deepgram/{df['Labels'][index]}/{df['Labels'][index]}.mp3"
  #getting the varaibles from the function
  transcript_json, transcript_txt = transcribe_audio(audio_path)

  #save path
  save_path=f"/content/drive/MyDrive/Emergency_Meeting_Deepgram/{df['Labels'][index]}"

  #text file
  new_file=f"{df['Labels'][index]}.txt"
  #text path
  text_path = os.path.join(save_path, new_file)

  #Writing the text to a file
  with open(text_path, 'w', encoding='utf-8') as text_file:
    text_file.write(transcript_txt)

  print(f" Text file saved to: {text_path}")

  #json file
  json_file =f"{df['Labels'][index]}.json"
  #json file path
  json_path = os.path.join(save_path, json_file)

  #Writing the JSON to a file
  with open(json_path, 'w', encoding='utf-8') as json_file:
    json.dump(transcript_json, json_file, indent=4)

  print(f"JSON saved to: {json_path}")


## Random Sample for Accuracy check

In [ ]:
df.sample(n=5)